# 06 — Bronze DLT Pipeline | CSV Ingestion (Chunk 2)

## Configuration

In [0]:
import dlt
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StructType, StructField, StringType

CATALOG = spark.conf.get("catalog_name", "vstone_catalog")
RAW_SCHEMA = spark.conf.get("raw_schema", "raw")
CHUNKS_VOL = spark.conf.get("chunks_volume", "chunks")
LANDING_VOL = spark.conf.get("landing_volume", "landing")

CHUNKS_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{CHUNKS_VOL}"
LANDING_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{LANDING_VOL}"

BRONZE_PROPS = {
    "quality": "bronze",
    "delta.enableChangeDataFeed": "true",
    "pipelines.reset.allowed": "true",
}

## Bronze Schema

In [0]:
BRONZE_SCHEMA = StructType([
    StructField("noise", StringType(), True),
    StructField("pollution", StringType(), True),
    StructField("date", StringType(), True),
    StructField("light", StringType(), True),
    StructField("raining", StringType(), True),
    StructField("street_id", StringType(), True),
])

print(f"Schema defined — {len(BRONZE_SCHEMA.fields)} columns, inferSchema = false (all STRING)")

## DLT Table —streets_csv__dlt

In [0]:
@dlt.table(
    name="streets_csv_dlt",
    comment="Bronze: streets.csv chunk 2 (20%, incremental) via Delta Live Tables. Grain: (street_id, date).",
)
def streets_csv_dlt():
    return (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .schema(BRONZE_SCHEMA)
            .load(f"{CHUNKS_PATH}/streets_chunk_2.csv")
            .withColumn("load_dt", current_timestamp())
            .withColumn("source", col("_metadata.file_name")))